# Decoder-layer representation audit — `decoder_layers_v1`

**One question.** Is PROB's final decoder layer the wrong feature space for semantic
novelty and clustering, and does an earlier layer do better?

**Inference only. No training.** One forward pass over the same fixed 1,600-image pool,
capturing all six decoder hidden states with forward hooks. ~20–40 min on a T4.

Protocol frozen before this ran: `docs/decoder_layer_protocol_2026-09-02.md`.
Decision rule is in code (`tools/audit_decoder_layers.py`), not in this notebook,
so it cannot drift between runs.

**Run all.** Nothing below needs editing except cell 1.

In [ ]:
# 1 — CONFIG. The only cell you may edit.
OWL_COMMIT   = "1b4129a8f96ead1e0780055ce4029a139ae48fc5"   # pinned; do not float to main
PROB_COMMIT  = "4c66be1a52cad9360e09c729e9134aba8fe0b531"
DRIVE_ROOT   = "/content/drive/MyDrive/OWL"
CHECKPOINT   = f"{DRIVE_ROOT}/checkpoints/SOWODB/t1.pth"
EXPORT_PATH  = f"{DRIVE_ROOT}/features/decoder_layers_v1.npz"
BATCH_SIZE   = 4
print("config pinned:", OWL_COMMIT[:8], PROB_COMMIT[:8])

## 2 — Drive, and prove the export directory is writable

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib
pathlib.Path(EXPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
probe = pathlib.Path(EXPORT_PATH).parent / ".write_probe"
probe.write_text("ok"); assert probe.read_text() == "ok"; probe.unlink()
assert os.path.exists(CHECKPOINT), f"checkpoint missing: {CHECKPOINT}"
print("drive ok; checkpoint", round(os.path.getsize(CHECKPOINT)/1e6), "MB")

## 3 — Pin OWL and PROB exactly

A floating clone is how a notebook silently changes what it measured. Both are pinned
and the resolved SHAs are printed and asserted.

In [ ]:
import subprocess, sys, os

def run(cmd, **kw):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)

if not os.path.exists("/content/owod-active"):
    run(["git","clone","-q","https://github.com/gubiczam/owod-active.git","/content/owod-active"])
run(["git","-C","/content/owod-active","fetch","-q","--all"])
run(["git","-C","/content/owod-active","checkout","-q",OWL_COMMIT])
owl_sha = subprocess.check_output(["git","-C","/content/owod-active","rev-parse","HEAD"]).decode().strip()
assert owl_sha.startswith(OWL_COMMIT[:8]) or OWL_COMMIT.startswith(owl_sha[:8]), (owl_sha, OWL_COMMIT)

if not os.path.exists("/content/PROB"):
    run(["git","clone","-q","https://github.com/gubiczam/PROB.git","/content/PROB"])
run(["git","-C","/content/PROB","fetch","-q","--all"])
run(["git","-C","/content/PROB","checkout","-q",PROB_COMMIT])
prob_sha = subprocess.check_output(["git","-C","/content/PROB","rev-parse","HEAD"]).decode().strip()
assert prob_sha == PROB_COMMIT, (prob_sha, PROB_COMMIT)

sys.path.insert(0, "/content/owod-active")
print("OWL ", owl_sha)
print("PROB", prob_sha)

## 4 — Dependencies, and the MSDA kernel

`MultiScaleDeformableAttention` is optional here: this is a single inference pass, so the
pure-PyTorch fallback costs minutes rather than hours. The build is attempted and the
outcome is *reported* rather than assumed, because a silent fallback is what made an
earlier session mis-price itself.

In [ ]:
run([sys.executable,"-m","pip","install","-q","-e","/content/owod-active"])
run([sys.executable,"-m","pip","install","-q","scikit-learn","scipy","matplotlib"])

msda = "unavailable (pure-PyTorch fallback, fine for one inference pass)"
try:
    subprocess.run([sys.executable,"setup.py","build","install"],
                   cwd="/content/PROB/models/ops", check=True,
                   capture_output=True, timeout=1800)
    import MultiScaleDeformableAttention  # noqa: F401
    msda = "compiled CUDA kernel"
except Exception as error:
    print("MSDA build did not succeed:", str(error)[:300])
print("MSDA:", msda)

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "this notebook needs a GPU runtime"

## 5 — Materialise exactly the pool's images

The committed annotation archive is extracted and the 1,600 JPEGs the pool names are
fetched from COCO's public bucket by id — the same mechanism the replay notebook uses, so
no new provenance is introduced. It writes `ImageSets/OWDETR/owl_layer_test.txt`, whose
name has `test` as its only marker; `owl.evaluation_subset.check_split_name` refuses
anything else, because PROB picks its annotation filters by *substring* of the split name.

Fails closed if any image cannot be fetched.

In [ ]:
DATA_ROOT = "/content/data/OWOD"
run([sys.executable, "/content/owod-active/tools/materialize_pool_images.py",
     "--data-root", DATA_ROOT])

import numpy as np, pathlib
pool_path = "/content/owod-active/data/pool/sowodb_t1_frozen_pool.npz"
payload = np.load(pool_path, allow_pickle=True)
keep = np.asarray(payload["split"], dtype=str) == "pool"
image_ids = sorted(set(np.asarray(payload["image_ids"], dtype=str)[keep].tolist()))
jpeg = pathlib.Path(DATA_ROOT) / "JPEGImages"
present = sum(1 for name in image_ids if (jpeg / f"{name}.jpg").exists())
print(f"{present}/{len(image_ids)} pool images on disk, {int(keep.sum()):,} proposals")
assert present == len(image_ids)

## 6 — Smoke test: the whole path on one image, gated

Everything the full export can get wrong — checkpoint, reconstructed model arguments, the
PROB working directory its backbone needs, the hooks, the transforms convention, the key
join — is exercised here on a single image and checked with **the real gate**: `hs[5]` for
that image's own pool rows must reproduce the committed embeddings. Seconds, not 25 minutes.

A shape check alone would pass while the join was silently wrong, which is why this runs
the gate rather than an assertion about tensor sizes.

In [ ]:
run([sys.executable, "/content/owod-active/tools/export_decoder_layers.py",
     "--prob-root","/content/PROB",
     "--data-root",DATA_ROOT,
     "--checkpoint",CHECKPOINT,
     "--pool",pool_path,
     "--out",EXPORT_PATH,
     "--smoke-images","1"])

## 7 — Export `hs[0..5]`

Resumable: if the export exists it is verified instead of recomputed. The gate is a single
assertion — `hs[5]` must reproduce the pool's committed embeddings — which validates the
checkpoint, the reconstructed model arguments, the hooks, the image order and the key join
at once.

In [ ]:
run([sys.executable, "/content/owod-active/tools/export_decoder_layers.py",
     "--prob-root","/content/PROB",
     "--data-root",DATA_ROOT,
     "--checkpoint",CHECKPOINT,
     "--pool",pool_path,
     "--out",EXPORT_PATH,
     "--batch-size",str(BATCH_SIZE)])

## 8 — The audit. Same protocol for every layer; the decision rule lives in the tool.

In [ ]:
run([sys.executable, "/content/owod-active/tools/audit_decoder_layers.py",
     "--export", EXPORT_PATH, "--seeds", "3"],
    cwd="/content/owod-active")

## 9 — Plots, then persist every result to Drive

In [ ]:
run([sys.executable, "/content/owod-active/tools/plot_decoder_layers.py"],
    cwd="/content/owod-active")

import shutil, pathlib, json
dest = pathlib.Path(DRIVE_ROOT) / "results" / "decoder_layer_audit"
dest.mkdir(parents=True, exist_ok=True)
src = pathlib.Path("/content/owod-active/data/results")
copied = []
for pattern in ("decoder_layer_*.csv", "decoder_layer_*.png"):
    for path in sorted(src.glob(pattern)):
        shutil.copy2(path, dest / path.name); copied.append(path.name)
(dest / "run_provenance.json").write_text(json.dumps({
    "owl_commit": owl_sha, "prob_commit": prob_sha, "msda": msda,
    "export": EXPORT_PATH, "checkpoint": CHECKPOINT, "data_root": DATA_ROOT,
    "images": len(image_ids), "proposals": int(keep.sum()),
}, indent=2), encoding="utf-8")
print("persisted to", dest)
for name in copied: print("  ", name)

## 10 — Verdict

Printed by cell 7 and recomputed here from the saved CSV so the notebook's last word and
the committed table cannot disagree.

In [ ]:
import sys
sys.path.insert(0,"/content/owod-active")
import csv
from tools.audit_decoder_layers import decide
rows = list(csv.DictReader(open("/content/owod-active/data/results/decoder_layer_population.csv")))
for row in rows:
    for key, value in list(row.items()):
        if key not in ("representation","population","novelty_sign_correct"):
            try: row[key] = float(value)
            except (TypeError, ValueError): pass
decision = decide(rows)
print("VERDICT:", decision["verdict"])
for entry in decision.get("layers", []):
    print(f"  layer {entry['layer']}: unk_knn={entry['unknown_knn']:.4f} "
          f"open={entry['open_pool']:.4f} auc={entry['auc']:.4f} "
          f"{'PASS' if entry['passes'] else 'fail'}")